[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gonzalezulises/formacion-docente-bi-faces/blob/main/modulo-03-business-intelligence/laboratorios/lab_05_dashboard_paso_a_paso.ipynb)

# Laboratorio 05 — Construir un dashboard completo en Power BI

**Módulo 3: Business Intelligence aplicado a decisiones**  
**Instructora:** Ilda Rojas  
**Duración estimada:** 60-90 minutos

---

## Objetivo

Construir un dashboard completo en Power BI Desktop utilizando el dataset `rendimiento_academico.csv`, aplicando todos los conceptos vistos en las sesiones anteriores: transformación de datos, medidas DAX, visualizaciones y narrativa.

## Requisitos previos

- Power BI Desktop instalado (o Tableau Public / Looker Studio como alternativa)
- Dataset: `../datasets/universidad/rendimiento_academico.csv`
- Haber completado las sesiones 03.01 a 03.04

## Entregable

Un archivo `.pbix` (Power BI) con un dashboard de una página que incluya al menos: 3 tarjetas KPI, 2 gráficos, 1 segmentador y 1 título con hallazgo.

---

## Paso 1: Importar y transformar datos en Power Query

### 1.1 Importar el archivo

1. Abrir Power BI Desktop
2. **Inicio → Obtener datos → Texto/CSV**
3. Seleccionar `rendimiento_academico.csv`
4. En la vista previa, verificar:
   - ¿Se detectaron correctamente las columnas?
   - ¿El separador es correcto (coma, punto y coma)?
   - ¿Los encabezados están en la primera fila?
5. Clic en **Transformar datos** (NO en "Cargar")

### 1.2 Transformaciones en Power Query

En el Editor de Power Query, realice las siguientes transformaciones:

| # | Transformación | Pasos |
|---|---|---|
| 1 | **Verificar tipos** | Clic en cada encabezado de columna y verificar que el ícono sea correcto: ABC (texto), 123 (número), calendario (fecha) |
| 2 | **Corregir tipos** | Si alguna columna numérica se importó como texto: clic en el ícono → Número decimal |
| 3 | **Renombrar columnas** | Doble clic en encabezados poco claros y renombrar (ej: `nota_final` → `Nota Final`) |
| 4 | **Revisar nulos** | Clic en la flecha de cada columna → verificar si hay (null). Decidir: ¿eliminar filas? ¿reemplazar valor? |
| 5 | **Eliminar duplicados** | Si hay filas duplicadas: seleccionar todas las columnas → Inicio → Quitar filas → Quitar duplicados |

### 1.3 Aplicar cambios

1. Revisar los "Pasos aplicados" en el panel derecho (cada transformación queda registrada)
2. Clic en **Cerrar y aplicar**

In [ ]:
# Referencia: estructura del dataset que importará en Power BI
import pandas as pd

df = pd.read_csv('../datasets/universidad/rendimiento_academico.csv')

print("ESTRUCTURA DEL DATASET")
print("="*50)
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
print()
print("Columnas y tipos:")
for col in df.columns:
    nulos = df[col].isnull().sum()
    nulos_str = f" ({nulos} nulos)" if nulos > 0 else ""
    print(f"  {col}: {df[col].dtype}{nulos_str}")
print()
print("Vista previa:")
df.head(3)

---

## Paso 2: Crear medidas DAX

En la vista de Informe (ícono de gráfico de barras en la barra lateral izquierda):

### Medida 1: Total de estudiantes

1. Clic derecho en la tabla en el panel de campos → **Nueva medida**
2. Escribir:

```dax
Total Estudiantes = DISTINCTCOUNT(rendimiento_academico[id_estudiante])
```

*(Si la columna se llama diferente, ajuste el nombre)*

### Medida 2: Promedio de notas

```dax
Promedio Notas = AVERAGE(rendimiento_academico[nota])
```

**Formato:** Clic en la medida → pestaña Medida → Formato: Número decimal, 1 decimal.

### Medida 3: Tasa de aprobación

```dax
Tasa Aprobación = 
    DIVIDE(
        CALCULATE(
            COUNT(rendimiento_academico[id_estudiante]),
            rendimiento_academico[estado] = "Aprobado"
        ),
        COUNT(rendimiento_academico[id_estudiante]),
        0
    )
```

**Formato:** Porcentaje, 1 decimal.

### Medida 4 (opcional): Cantidad de carreras

```dax
Carreras = DISTINCTCOUNT(rendimiento_academico[carrera])
```

> **Tip:** Después de crear cada medida, verifique que aparezca con el ícono de calculadora (fx) en el panel de campos.

---

## Paso 3: Diseñar el layout

Antes de crear visualizaciones, planifique la distribución de la página.

### Layout recomendado

```
┌─────────────────────────────────────────────────────────────────┐
│                                                                 │
│   TÍTULO: Dashboard de Rendimiento Académico — FACES            │
│   Subtítulo: Período [X] | Actualizado: [fecha]                 │
│                                                                 │
├───────────┬───────────┬───────────┬─────────────────────────────┤
│           │           │           │                             │
│  TARJETA  │  TARJETA  │  TARJETA  │   SEGMENTADOR              │
│  Total    │  Promedio │  Tasa     │   Carrera                  │
│  Estud.   │  Notas    │  Aprob.   │   [dropdown]               │
│           │           │           │                             │
├───────────┴───────────┴───────────┼────────────┬────────────────┤
│                                   │            │                │
│   GRÁFICO DE BARRAS               │ SEGMENTADOR│                │
│   Estudiantes por carrera         │ Período    │                │
│   (horizontal, ordenado desc.)    │ [lista]    │                │
│                                   │            │                │
├───────────────────────────────────┼────────────┴────────────────┤
│                                   │                             │
│   GRÁFICO DE LÍNEAS              │   TABLA                     │
│   Promedio por período            │   Resumen por carrera       │
│   (con línea de meta)             │   (carrera, N, prom, tasa)  │
│                                   │                             │
└───────────────────────────────────┴─────────────────────────────┘
```

### Guía de tamaños (en una página 1280x720)

| Elemento | Ancho | Alto | Posición |
|---|---|---|---|
| Título | 100% | 60px | Arriba |
| Tarjetas (x3) | 200px c/u | 100px | Debajo del título, izquierda |
| Segmentador carrera | 300px | 100px | Arriba derecha |
| Barras | 60% | 250px | Centro izquierda |
| Líneas | 60% | 250px | Abajo izquierda |
| Tabla | 40% | 250px | Abajo derecha |

---

## Paso 4: Agregar interactividad

### 4.1 Segmentadores

**Segmentador de carrera:**
1. Insertar → Segmentador
2. Arrastrar `carrera` al campo
3. Formato → Estilo: **Dropdown** (menú desplegable)
4. Activar "Selección múltiple" si desea filtrar por varias carreras

**Segmentador de período:**
1. Insertar → Segmentador
2. Arrastrar `periodo` al campo
3. Formato → Estilo: **Lista** o **Between** (rango)

### 4.2 Interacciones entre visualizaciones

Por defecto, Power BI aplica cross-filtering. Para personalizar:

1. Seleccionar el gráfico de barras
2. Ir a **Formato → Editar interacciones**
3. En cada otro gráfico, aparecen íconos:
   - Embudo: filtrar
   - Resaltar: destacar sin filtrar
   - Prohibido: no interactuar

**Configuración recomendada:**
- Barras → Líneas: **Filtrar** (al clic en carrera, la línea muestra solo esa carrera)
- Barras → Tabla: **Filtrar**
- Barras → Tarjetas: **Filtrar** (los KPIs se recalculan para la carrera seleccionada)

### 4.3 Drill-down por carrera/período

Para agregar jerarquía de drill-down:

1. En el gráfico de barras, agregar `periodo` como segundo nivel debajo de `carrera`
2. Aparecen flechas de drill en el encabezado del gráfico:
   - ↓ Ir al siguiente nivel
   - ↑ Volver al nivel anterior
   - ⇊ Expandir todo el siguiente nivel

### 4.4 Tooltips personalizados

1. Seleccionar un gráfico
2. En la tarjeta Marcas, arrastrar campos adicionales a **Tooltip**
3. Estos campos aparecerán al pasar el mouse sobre el gráfico
4. Personalizar el formato: Formato → Tooltip → editar texto

---

## Paso 5: Exportar y compartir

### 5.1 Guardar el archivo

1. **Archivo → Guardar como** → nombre descriptivo: `dashboard_rendimiento_faces.pbix`

### 5.2 Exportar a PDF

1. **Archivo → Exportar → PDF**
2. Seleccionar páginas a exportar
3. Útil para enviar por correo o imprimir

### 5.3 Publicar en Power BI Service

1. **Inicio → Publicar**
2. Seleccionar el área de trabajo (requiere cuenta Pro o Premium)
3. Compartir enlace con los interesados

### 5.4 Exportar datos

1. Clic derecho en cualquier visualización → **Exportar datos**
2. Exporta los datos subyacentes a CSV o Excel

In [ ]:
# Vista previa final: lo que debería mostrar su dashboard
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('../datasets/universidad/rendimiento_academico.csv')

fig = plt.figure(figsize=(16, 10))
fig.suptitle('Dashboard de Rendimiento Académico — FACES', 
             fontsize=16, fontweight='bold', y=0.98)

# KPIs
ax_kpi = fig.add_axes([0.05, 0.85, 0.9, 0.08])
ax_kpi.axis('off')

if 'nota' in df.columns:
    total = df['id_estudiante'].nunique() if 'id_estudiante' in df.columns else len(df)
    prom = df['nota'].mean()
    tasa = (df['estado'] == 'Aprobado').mean() * 100 if 'estado' in df.columns else 0
    
    ax_kpi.text(0.17, 0.5, f'Total Estudiantes\n{total:,}', 
                ha='center', va='center', fontsize=14, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.5', facecolor='#dbeafe', edgecolor='#2563eb'))
    ax_kpi.text(0.5, 0.5, f'Promedio Notas\n{prom:.1f}', 
                ha='center', va='center', fontsize=14, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.5', facecolor='#dbeafe', edgecolor='#2563eb'))
    ax_kpi.text(0.83, 0.5, f'Tasa Aprobación\n{tasa:.1f}%', 
                ha='center', va='center', fontsize=14, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.5', facecolor='#dbeafe', edgecolor='#2563eb'))

# Barras
ax1 = fig.add_subplot(2, 2, 3)
if 'carrera' in df.columns:
    df.groupby('carrera')['nota'].mean().sort_values().plot(kind='barh', ax=ax1, color='#2563eb')
    ax1.set_title('Promedio por carrera', fontweight='bold')
    ax1.set_xlabel('Promedio')

# Líneas
ax2 = fig.add_subplot(2, 2, 4)
if 'periodo' in df.columns:
    df.groupby('periodo')['nota'].mean().plot(ax=ax2, marker='o', color='#2563eb', linewidth=2)
    ax2.axhline(y=14, color='red', linestyle='--', alpha=0.5, label='Meta')
    ax2.set_title('Tendencia por período', fontweight='bold')
    ax2.legend()
    ax2.tick_params(axis='x', rotation=45)

plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()

---

## Checklist de evaluación

Verifique que su dashboard cumple con estos criterios:

| # | Criterio | Cumple |
|---|---|---|
| 1 | Datos importados y tipos correctos | ☐ |
| 2 | Al menos 3 medidas DAX creadas | ☐ |
| 3 | Al menos 3 tarjetas KPI | ☐ |
| 4 | Al menos 1 gráfico de barras | ☐ |
| 5 | Al menos 1 gráfico de líneas o tabla | ☐ |
| 6 | Al menos 1 segmentador funcional | ☐ |
| 7 | Título descriptivo (no genérico) | ☐ |
| 8 | Colores consistentes (paleta definida) | ☐ |
| 9 | Sin chartjunk (bordes, sombras, 3D innecesarios) | ☐ |
| 10 | Las interacciones entre gráficos funcionan correctamente | ☐ |